# Quantum Error Correction Surface Code Simulator

This notebook demonstrates the core functionality of the **Quantum Error Correction Surface Code Simulator**.

The project implements a modular framework for simulating planar surface codes using **Stim** for fast stabilizer circuit simulation and **PyMatching** for Minimum Weight Perfect Matching (MWPM) decoding.

The notebook walks through the complete simulation pipeline:

1. Construct a surface-code circuit
2. Generate a Detector Error Model (DEM)
3. Sample detector events
4. Decode syndromes using PyMatching
5. Estimate logical failure rates

The implementation currently supports configurable code distances, multi-round syndrome extraction, configurable noise models, and both **X** and **Z** logical memory experiments.

## Imports

Import the simulator backend together with the decoding and simulation utilities used throughout this demonstration.

In [1]:
from qec.stim_backend import SurfaceCodeStimBackend
from qec.decoders.mwpm import MWPMDecoder
from qec.simulation import logical_failure_rate_stim

import stim
import numpy as np

## Build a Surface-Code Circuit

The `SurfaceCodeStimBackend` constructs a Stim-compatible circuit representing repeated syndrome extraction for a planar surface code.

In this example we generate a distance-3 code with five rounds of syndrome extraction. The circuit is configured to perform a **Z-memory experiment**, where the logical information is encoded and measured in the computational basis.

In [2]:
backend = SurfaceCodeStimBackend(
    distance=3,
    rounds=5,
    depolarizing_error=0.001,
    readout_error=0.001,
    memory_basis="Z",
)

circuit = backend.build_circuit()

print(f"Distance          : {backend.distance}")
print(f"Syndrome rounds   : {backend.rounds}")
print(f"Data qubits       : {backend.n_data}")
print(f"Stabilizers       : {backend.n_stabilizers}")
print(f"Total qubits      : {circuit.num_qubits}")

Distance          : 3
Syndrome rounds   : 5
Data qubits       : 9
Stabilizers       : 4
Total qubits      : 13


## Inspecting the Stim Circuit

The backend generates a complete **Stim** circuit describing repeated stabilizer measurements, detector definitions, and logical observables.

Each round consists of:

- Ancilla qubit reset
- Stabilizer measurements
- Noise injection
- Detector construction

The final round measures the data qubits and defines the logical observable corresponding to the chosen memory basis.

In [3]:
print(circuit)

R 0 1 2 3 4 5 6 7 8 9 10 11 12 9 10 11 12
H 9
CX 9 0 9 1 9 3 9 4
H 9
X_ERROR(0.001) 9
M 9
CX 1 10 2 10 4 10 5 10
X_ERROR(0.001) 10
M 10
CX 3 11 4 11 6 11 7 11
X_ERROR(0.001) 11
M 11
H 12
CX 12 4 12 5 12 7 12 8
H 12
X_ERROR(0.001) 12
M 12
DEPOLARIZE1(0.001) 0 1 2 3 4 5 6 7 8
R 9 10 11 12
H 9
CX 9 0 9 1 9 3 9 4
H 9
X_ERROR(0.001) 9
M 9
CX 1 10 2 10 4 10 5 10
X_ERROR(0.001) 10
M 10
CX 3 11 4 11 6 11 7 11
X_ERROR(0.001) 11
M 11
H 12
CX 12 4 12 5 12 7 12 8
H 12
X_ERROR(0.001) 12
M 12
DEPOLARIZE1(0.001) 0 1 2 3 4 5 6 7 8
DETECTOR(0.5, 0.5, 1, 0) rec[-8] rec[-4]
DETECTOR(0.5, 1.5, 1, 1) rec[-7] rec[-3]
DETECTOR(1.5, 0.5, 1, 1) rec[-6] rec[-2]
DETECTOR(1.5, 1.5, 1, 0) rec[-5] rec[-1]
R 9 10 11 12
H 9
CX 9 0 9 1 9 3 9 4
H 9
X_ERROR(0.001) 9
M 9
CX 1 10 2 10 4 10 5 10
X_ERROR(0.001) 10
M 10
CX 3 11 4 11 6 11 7 11
X_ERROR(0.001) 11
M 11
H 12
CX 12 4 12 5 12 7 12 8
H 12
X_ERROR(0.001) 12
M 12
DEPOLARIZE1(0.001) 0 1 2 3 4 5 6 7 8
DETECTOR(0.5, 0.5, 2, 0) rec[-8] rec[-4]
DETECTOR(0.5, 1.5, 2, 1) rec

## Detector Error Model

Stim can automatically convert a quantum circuit into a **Detector Error Model (DEM)**.

Rather than describing the circuit itself, the DEM describes how physical errors propagate to detector outcomes and logical observables. This compact representation is used directly by PyMatching to construct a Minimum Weight Perfect Matching decoder.

In [4]:
dem = backend.detector_error_model()

print(dem)

error(0.001997667259370424) D0
error(0.0003334445185803059) D0 D1
error(0.0003334445185803059) D0 D1 D2 D3
error(0.0003334445185803059) D0 D2 L0
error(0.0003334445185803059) D0 D3
error(0.001) D0 D4
error(0.0003334445185803059) D0 L0
error(0.00232977955555556) D1
error(0.0003334445185803059) D1 D2
error(0.0003334445185803059) D1 D3
error(0.001) D1 D5
error(0.001332777629543145) D2
error(0.0003334445185803059) D2 D3
error(0.001) D2 D6
error(0.0009996665925555348) D2 L0
error(0.00232977955555556) D3
error(0.001) D3 D7
error(0.0009996665925555348) D4
error(0.0003334445185803059) D4 D5
error(0.0003334445185803059) D4 D5 D6 D7
error(0.0003334445185803059) D4 D6 L0
error(0.0003334445185803059) D4 D7
error(0.001) D4 D8
error(0.0003334445185803059) D4 L0
error(0.001332444444444449) D5
error(0.0003334445185803059) D5 D6
error(0.0003334445185803059) D5 D7
error(0.001) D5 D9
error(0.0003334445185803059) D6
error(0.0003334445185803059) D6 D7
error(0.001) D6 D10
error(0.0009996665925555348) D6 L0
e

## Detector Sampling

Once the circuit has been compiled, Stim can efficiently sample detector events generated by noisy executions of the circuit.

Each sampled detector vector records which stabilizer checks detected an inconsistency during syndrome extraction.

In [15]:
detectors = backend.sample_detectors(
    shots=5,
)

print("Detector sample shape:", detectors.shape)

print("\nFirst detector sample:\n")
print(detectors[0].astype(int))

Detector sample shape: (5, 18)

First detector sample:

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


## Decoding with PyMatching

The detector error model can be converted directly into a **PyMatching** decoder.

Given a set of detector events, the decoder predicts whether a logical error has occurred by solving a Minimum Weight Perfect Matching (MWPM) problem on the detector graph.

In [6]:
decoder = MWPMDecoder(
    backend="pymatching",
    dem=dem,
)

decoder.matching

<pymatching.Matching object with 18 detectors, 0 boundary nodes, and 57 edges>

## Decode a Detector Sample

The detector events generated by Stim can be passed directly to the decoder.

The decoder returns the predicted logical observable for the sampled error configuration.

In [7]:
prediction = decoder.decode_detection_events(
    detectors[0]
)

print("Detector events:")
print(detectors[0].astype(int))

print("\nPredicted logical observable:")
print(prediction)

Detector events:
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

Predicted logical observable:
[0]


## Logical Failure-Rate Simulation

A complete logical memory experiment consists of repeatedly:

1. Building the noisy surface-code circuit.
2. Sampling detector events using Stim.
3. Decoding with PyMatching.
4. Comparing the decoder prediction with the measured logical observable.

The logical failure rate is estimated as the fraction of shots for which the decoder prediction differs from the measured logical observable.

In [14]:
physical_error_rate = 0.01

logical_error_rate = logical_failure_rate_stim(
    distance=3,
    rounds=5,
    shots=1000,
    depolarizing_error=physical_error_rate,
    readout_error=physical_error_rate,
    memory_basis="Z",
)

print(
    f"Physical error rate : {physical_error_rate:.3f}"
)

print(
    f"Logical failure rate: {logical_error_rate:.4f}"
)

Physical error rate : 0.010
Logical failure rate: 0.1020


## Comparing Code Distances

Increasing the code distance increases the number of physical qubits used to encode the logical qubit.

The following example estimates the logical failure rate for several code distances using a small number of Monte Carlo samples. This is intended as a demonstration of the simulation workflow rather than a comprehensive benchmark.

In [12]:
backend = SurfaceCodeStimBackend(
    distance=3,
    rounds=5,
    memory_basis="Z",
)

detectors, observables = (
    backend.sample_detectors_and_observables(
        shots=3
    )
)

print("Detector shape:", detectors.shape)
print("Observable shape:", observables.shape)

print("\nFirst logical measurement:")
print(observables[0])

Detector shape: (3, 18)
Observable shape: (3, 1)

First logical measurement:
[False]


## Running Full Experiments

The notebook demonstrates the complete simulation pipeline on small examples.

For larger Monte Carlo studies and figure generation, the repository provides dedicated experiment scripts:

- `experiments/logical_failure_stim.py`
- `experiments/compare_decoders.py`

These scripts generate publication-quality figures and perform substantially larger simulations than are appropriate for an interactive notebook.

# Summary

In this notebook we have demonstrated the complete quantum error correction workflow implemented in this project:

- Constructed a planar surface-code circuit using Stim
- Generated a Detector Error Model (DEM)
- Sampled detector events from noisy circuit executions
- Decoded syndromes using the PyMatching MWPM decoder
- Estimated logical failure rates through Monte Carlo simulation

The simulator currently supports configurable code distances, multiple rounds of syndrome extraction, configurable noise models, and both X- and Z-memory experiments.

The next development milestone is replacing the simplified checkerboard stabilizer layout with a physically accurate planar surface-code geometry and validating the expected improvement in logical error rate with increasing code distance.